# Walmart Retail Time Series Forecasting

In this exercise we use an **LSTM** to forecast item-level sales from the Walmart M5 competition dataset.

**Dataset:** `walmart_retail/` directory containing:
- `sales_train_validation.csv` -- daily unit sales per item per store (1,913 days)
- `calendar.csv` -- date-to-day mapping with event info
- `sell_prices.csv` -- weekly item prices (not used here)
- `sales_train_evaluation.csv` -- extended sales data (not used here)

**Prepared subset.** The sales file is 120 MB. Its first 3,049 rows are all store CA_1, so this notebook reads only the first three rows -- exactly the three items analysed below -- and loads in about a second. Set `SUBSET_ROWS = None` in Section 2 to explore the full file.

**Learning objectives:**
1. Explore a real-world retail time series dataset.
2. Select and visualize individual item/store sales trajectories.
3. Prepare sequential data for LSTM forecasting with a training / validation / test split in time.
4. Build, train (with early stopping on validation) and evaluate an LSTM sales forecaster.
5. Compare it with the weekly seasonal naive and a 28-day mean on the same test days.

**Evaluation design.** Windows are split by target day: the first 70% for training, the next 10% for validation (early stopping and every comparison during development) and the last 20% for the final test, used once. Many days have zero sales, so we report MAE (and RMSE), not MAPE. Runtime: under half a minute on an Apple M3 Max laptop CPU; allow up to twice as long on older laptops.

---
## 1. Imports

In [ ]:
import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)  # initial weights, dropout and DataLoader shuffling

print(f"PyTorch version: {torch.__version__}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

---
## 2. Load and Explore the Data

In [ ]:
# Prepared subset: the first three rows are the three CA_1 items analysed below
SUBSET_ROWS = 3  # None loads all 30,490 rows (slow, about 0.5 GB of memory)
sales = pd.read_csv("walmart_retail/sales_train_validation.csv", nrows=SUBSET_ROWS)
print(f"Sales shape: {sales.shape}")
print(f"\nFirst 10 columns: {list(sales.columns[:10])}")
print(f"Last 5 columns:  {list(sales.columns[-5:])}")
sales.head(3)

In [ ]:
calendar = pd.read_csv("walmart_retail/calendar.csv")
print(f"Calendar shape: {calendar.shape}")
calendar.head()

In [ ]:
# Identify the day columns (d_1, d_2, ... d_1913)
day_cols = [c for c in sales.columns if c.startswith("d_")]
print(f"Number of day columns: {len(day_cols)}")
# These counts describe the rows loaded; the full file has 3,049 items in 10 stores
print(f"Number of unique items: {sales['item_id'].nunique()}")
print(f"Number of unique stores: {sales['store_id'].nunique()}")

---
## 3. Select Items and Stores for Analysis

The full dataset is very large. We will select a few specific items from specific stores to keep things manageable: the first three items of store CA_1.

In [ ]:
# Select 3 items from store CA_1
store = "CA_1"
store_data = sales[sales["store_id"] == store].head(3)

print(f"Selected items from {store}:")
for _, row in store_data.iterrows():
    print(f"  {row['id']}  (category: {row['cat_id']}, dept: {row['dept_id']})")

# Extract daily sales as numpy array
item_sales = store_data[day_cols].values.astype(np.float32)
item_names = store_data["id"].values
print(f"\nDaily sales matrix shape: {item_sales.shape}")

---
## 4. Visualize Sales Over Time

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

for i, (ax, name) in enumerate(zip(axes, item_names)):
    ax.plot(item_sales[i], linewidth=0.5)
    ax.set_title(name)
    ax.set_ylabel("Units Sold")
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Day")
plt.suptitle(f"Daily Sales for Selected Items ({store})", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Use 7-day rolling average to see trend
fig, ax = plt.subplots(figsize=(14, 4))
for i, name in enumerate(item_names):
    rolling = pd.Series(item_sales[i]).rolling(window=7).mean()
    ax.plot(rolling, label=name, linewidth=0.8)
ax.set_title("7-Day Rolling Average Sales")
ax.set_xlabel("Day")
ax.set_ylabel("Avg Units Sold")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 5. Prepare Data: Focus on One Item

We will train an LSTM to forecast sales for the first selected item, one day ahead from the past 28 days.

Each window belongs to the period that contains its **target** day: the first 70% of windows for training, the next 10% for validation and the last 20% for the test. Validation and test windows may read the preceding 28 observed days as inputs -- that is what a deployed forecast sees -- but no training target lies after the training period. The scaler is fitted on the days used by the training windows only.

In [ ]:
# Focus on item 0
series = item_sales[0].reshape(-1, 1)
print(f"Series length: {len(series)}")
print(f"Item: {item_names[0]}")
print(f"Days with zero sales: {(series == 0).mean():.0%}")

# Normalize
scaler = MinMaxScaler(feature_range=(0, 1))
# Match the later 70/10/20 chronological split of one-step windows.
SEQ_LENGTH = 28
train_sequence_count = int((len(series) - SEQ_LENGTH) * 0.7)
training_raw_end = SEQ_LENGTH + train_sequence_count
scaler.fit(series[:training_raw_end])  # training days only
series_scaled = scaler.transform(series)
# Later observed values may fall outside [0, 1]; do not refit or clip them.
print(f"Scaler fitted on days 1-{training_raw_end} of {len(series)}")

In [ ]:
def create_sequences(data, seq_length):
    """Sliding window to create (input, target) pairs."""
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i : i + seq_length])
        y.append(data[i + seq_length])
    return np.array(X), np.array(y)


SEQ_LENGTH = 28  # use past 28 days (4 weeks) to predict next day

X, y = create_sequences(series_scaled, SEQ_LENGTH)
print(f"X shape: {X.shape}, y shape: {y.shape}")

# Chronological split by target day: 70% train, 10% validation, 20% test
train_end = int(len(X) * 0.7)
val_end = int(len(X) * 0.8)
assert train_end + SEQ_LENGTH == training_raw_end  # the scaler saw training days only
X_train, y_train = X[:train_end], y[:train_end]
X_val, y_val = X[train_end:val_end], y[train_end:val_end]
X_test, y_test = X[val_end:], y[val_end:]

print(f"Train: {X_train.shape[0]}, Validation: {X_val.shape[0]}, Test: {X_test.shape[0]}")

In [ ]:
# PyTorch tensors and DataLoader
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32)

BATCH_SIZE = 32
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t),
                          batch_size=BATCH_SIZE, shuffle=True)

### Baselines on the same days

Simple forecasts read from the same input windows, so they have exactly the same targets as the LSTM:
- **Seasonal naive (7 days)**: the sales on the same weekday one week before the target, `X[:, -7]`.
- **28-day mean**: the mean of the 28 input days.
- **Zero (no sales)**: worth including for intermittent sales. MAE rewards the median, and this item sells nothing on most days; the LSTM, trained with MSE, aims at the mean instead.

We score the validation days now; the test days are scored once, in Section 8.

In [ ]:
def to_units(z):
    """Invert the MinMaxScaler: scaled values back to units sold."""
    return scaler.inverse_transform(np.asarray(z).reshape(-1, 1)).ravel()


def score(y_true, y_pred):
    return {"MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_true, y_pred))}


def baseline_forecasts(X):
    """Baseline forecasts in units sold, one per window."""
    return {"Seasonal naive (7 days)": to_units(X[:, -7, 0]),
            "28-day mean": to_units(X[:, :, 0].mean(axis=1)),
            "Zero (no sales)": np.zeros(len(X))}


y_val_units = to_units(y_val)
val_scores = {name: score(y_val_units, pred) for name, pred in baseline_forecasts(X_val).items()}
pd.DataFrame(val_scores).T.round(3)

---
## 6. Build LSTM for Sales Forecasting

In [ ]:
class SalesLSTM(nn.Module):
    """LSTM for univariate sales forecasting."""

    def __init__(self, input_size=1, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout,
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        return self.fc(lstm_out[:, -1, :])


model = SalesLSTM().to(DEVICE)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

---
## 7. Train and Evaluate

Train on the training windows and stop early when the validation loss has not improved for `PATIENCE` epochs; the weights of the best epoch (a deep copy) are restored before any evaluation. Every comparison in this section uses the validation days.

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

NUM_EPOCHS = 100
PATIENCE = 10
train_losses, val_losses = [], []
best_val_loss, best_epoch, best_state, patience_counter = float("inf"), 0, None, 0

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        preds = model(X_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * X_batch.size(0)
    train_losses.append(running_loss / len(X_train_t))

    # The validation loss decides when to stop
    model.eval()
    with torch.no_grad():
        val_loss = criterion(model(X_val_t.to(DEVICE)), y_val_t.to(DEVICE)).item()
    val_losses.append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss, best_epoch = val_loss, epoch
        best_state = copy.deepcopy(model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1

    if epoch % 20 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{NUM_EPOCHS}  Train: {train_losses[-1]:.6f}  Val: {val_loss:.6f}")
    if patience_counter >= PATIENCE:
        print(f"Early stopping at epoch {epoch}")
        break

model.load_state_dict(best_state)  # restore the best weights before evaluation
print(f"Restored epoch {best_epoch} with val loss: {best_val_loss:.6f}")

In [ ]:
epochs = range(1, len(train_losses) + 1)
plt.figure(figsize=(8, 4))
plt.plot(epochs, train_losses, label="Train Loss")
plt.plot(epochs, val_losses, label="Validation Loss")
plt.axvline(best_epoch, color="gray", linestyle="--", label=f"Chosen epoch ({best_epoch})")
plt.title("Training and Validation Loss (MSE)")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
def predict_units(model, X):
    """Forecasts in units sold for an array of windows."""
    model.eval()
    with torch.no_grad():
        pred = model(torch.tensor(X, dtype=torch.float32).to(DEVICE)).cpu().numpy()
    return to_units(pred)


# Development comparison on the validation days
val_scores["LSTM"] = score(y_val_units, predict_units(model, X_val))
pd.DataFrame(val_scores).T.round(3)

---
## 8. Final Test Evaluation and Predictions

This is the final evaluation, run once after every choice was made on validation: the baselines and the LSTM are scored on the same test days.

In [ ]:
y_actual = to_units(y_test)
test_forecasts = baseline_forecasts(X_test)
test_forecasts["LSTM"] = predict_units(model, X_test)
y_pred = test_forecasts["LSTM"]

test_scores = {name: score(y_actual, pred) for name, pred in test_forecasts.items()}
print(f"Test days: {len(y_actual)}")
pd.DataFrame(test_scores).T.round(3)

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(y_actual, label="Actual", linewidth=0.8)
plt.plot(y_pred, label="Predicted", linewidth=0.8, alpha=0.8)
plt.title(f"Sales Forecast: {item_names[0]}")
plt.xlabel("Day (test set)")
plt.ylabel("Units Sold")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 9. Summary

In this notebook we:

1. Loaded a prepared subset of the Walmart M5 retail dataset and explored its structure.
2. Selected specific items from a single store for analysis.
3. Visualized raw daily sales and 7-day rolling averages.
4. Prepared sliding-window sequences (28 days to predict 1 day ahead), split by target day into training, validation and test.
5. Built a 2-layer LSTM and trained it with early stopping on the validation loss.
6. Compared it with the weekly seasonal naive, a 28-day mean and a zero forecast -- on validation during development, and once on the test days.

**Key takeaways:**
- Retail sales data can be very noisy with many zero-sales days; MAPE is undefined there, so use MAE.
- Simple baselines are hard to beat on intermittent single-item sales. Say so plainly when it happens.
- MAE rewards the median: for an item that sells nothing on most days, a zero forecast can win on MAE. Choose the metric to match the decision the forecast supports.
- Caveat: this item sells nothing in its first 901 days, and `sell_prices.csv` lists no price for it before week 11325 (day 897): it was not yet on sale. A large part of the training period is therefore structural zeros, not demand.
- The sliding-window length should reflect the periodicity of the data (e.g., weekly patterns -> at least 7 days).
- More sophisticated approaches could incorporate calendar features (holidays, weekdays), price data, and hierarchical structure.
- For production forecasting, consider ensembling with traditional methods (e.g., ARIMA, exponential smoothing).